## Bike Sales Data Cleaning

**Goal:** Clean a messy retail bike sales export (typos, inconsistent formatting, missing values, duplicates) and prepare it for analysis using pandas.

**Dataset:** Raw Excel export of bike sales transactions (`uncleaned bike sales data.xlsx`) containing customer demographics, order details, and pricing.

**Issues identified in the raw data:**
- Typos in categorical values (e.g. `"Decmber"` instead of `"December"`)
- Inconsistent gender labels (`"M"`/`"W"` vs full words)
- Extra whitespace in column names and country values
- Inconsistent country/state naming (e.g. double-spaced `"United  States"`)
- Missing product descriptions
- Missing unit cost / unit price values
- Duplicate rows
- No age grouping or day-level date breakdown for analysis


## 1. Setup and Import

In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_row', None)
pd.set_option('display.max_column', None)

In [3]:
df = pd.read_excel(r"C:\Users\PC\Downloads\uncleaned bike sales data.xlsx")
df.head()

,Sales_Order #,Date,Day,Month,Year,Customer_Age,Age_Group,Customer_Gender,Country,State,Product_Category,Sub_Category,Product_Description,Order_Quantity,Unit_Cost,Unit_Price,Profit,Cost,Revenue
0,261695,2021-12-01,NaN,December,2021,39,Adults (35-64),F,United States,California,Bikes,Mountain Bikes,"Mountain-200 Black, 46",4,1252,2295,4172,5008,9180
1,261695,2021-12-01,1.0,December,2021,44,Adults (35-64),M,United Kingdom,England,Bikes,Mountain Bikes,"Mountain-200 Silver, 42",1,1266,2320,1054,1266,2320
2,261697,2021-12-02,2.0,December,2021,37,Adults (35-64),M,United States,California,Bikes,Mountain Bikes,"Mountain-400-W Silver, 46",2,420,769,698,840,1538
3,261698,2021-12-02,2.0,December,2021,31,Young Adults (25-34),F,Australia,New South Wales,Bikes,Mountain Bikes,"Mountain-400-W Silver, 42",1,420,769,349,420,769
4,261699,2021-12-03,3.0,December,2021,37,Adults (35-64),F,United States,California,Bikes,Mountain Bikes,"Mountain-200 Black, 46",2,0,2295,2086,0,4590


## 2. Baseline: Data Quality Before Cleaning


In [4]:
print("Shape:", df.shape)
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

Shape: (89, 19)

Missing values per column:
Sales_Order #          0
Date                   0
Day                    2
Month                  0
Year                   0
Customer_Age           0
Age_Group              1
Customer_Gender        0
Country                0
State                  0
Product_Category       0
Sub_Category           0
Product_Description    1
Order_Quantity         0
 Unit_Cost             0
 Unit_Price            0
 Profit                0
 Cost                  0
Revenue                0
dtype: int64

Duplicate rows: 0


## 3. Fix Column Names and Categorical Typos

In [6]:
df.columns = df.columns.str.strip()

# Fix known typo in Month column
df['Month'] = df['Month'].str.replace('Decmber', 'December')

## 4. Date Handling
Convert `Date` to a proper datetime type and derive `Day` for finer-grained time analysis.

In [7]:
df['Date'] = pd.to_datetime(df['Date'])
df['Day'] = df['Date'].dt.day

## 5. Age Grouping
Bin `Customer_Age` into standard marketing age brackets.

In [8]:
bins = [0, 24, 34, 64, float('inf')]
labels = ['Youth (<25)', 'Young Adults (25-34)', 'Adults (35-64)', 'Seniors (65+)']

df['Age_Group'] = pd.cut(df['Customer_Age'], bins=bins, labels=labels, right=True)
df['Age_Group'].value_counts()

Age_Group
Adults (35-64)          48
Young Adults (25-34)    31
Youth (<25)             10
Seniors (65+)            0
Name: count, dtype: int64

## 6. Standardize Gender Labels

In [9]:
df['Customer_Gender'] = df['Customer_Gender'].str.replace('F', 'Female')
df['Customer_Gender'] = df['Customer_Gender'].str.replace('M', 'Male')
df['Customer_Gender'].value_counts()

Customer_Gender
Female    50
Male      39
Name: count, dtype: int64

## 7. Clean Country and State Values

In [10]:
df['Country'] = df['Country'].str.strip()
df['Country'] = df['Country'].str.replace('United  States', 'United States')
print(df['Country'].value_counts())

Country
United States     33
Australia         27
United Kingdom     9
France             8
Germany            6
Canada             6
Name: count, dtype: int64


In [11]:
df['State'] = df['State'].str.replace(r'\bNord\b', 'Nordrhein-Westfalen', regex=True)
print(df['State'].value_counts())

State
California             20
New South Wales        14
England                 9
Washington              9
Queensland              6
British Columbia        6
Victoria                6
Nordrhein-Westfalen     4
Oregon                  4
Seine (Paris)           4
Hessen                  2
Hamburg                 1
Seine et Marne          1
Seine Saint Denis       1
South Australia         1
Somme                   1
Name: count, dtype: int64


## 8. Handle Missing Product Descriptions

In [12]:
df['Product_Description'] = df['Product_Description'].fillna('Mountain-200 Black, 46')

## 9. Impute Missing Cost/Price by Product Median
Rather than filling missing `Unit_Cost` / `Unit_Price` with a single global average (which would distort pricing across very different products), impute using the **median within each product group**. This preserves realistic per-product pricing.

In [13]:
df['Unit_Cost'] = df.groupby('Product_Description')['Unit_Cost'].transform('median')
df['Unit_Price'] = df.groupby('Product_Description')['Unit_Price'].transform('median')

## 10. Derive Financial Metrics

In [15]:
df['Cost'] = df['Unit_Cost'] * df['Order_Quantity']
df['Revenue'] = df['Unit_Price'] * df['Order_Quantity']
df['Profit'] = df['Revenue'] - df['Cost']

## 11. Quick Insight: Profit by Country
A sanity check that also doubles as a first analytical takeaway from the cleaned data.

In [16]:
df.groupby('Country')['Profit'].sum().sort_values(ascending=False)

Country
United States     61413.0
Australia         50326.0
France            20981.0
Germany           13636.0
Canada             9123.0
United Kingdom     9072.0
Name: Profit, dtype: float64

## 12. Remove Duplicates

In [17]:
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]
print(f"Removed {before - after} duplicate rows")
print(df.shape)

Removed 1 duplicate rows
(88, 19)


## 13. Final Data Quality Check

In [18]:
print(df.info())
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

<class 'pandas.DataFrame'>
Index: 88 entries, 0 to 88
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Sales_Order #        88 non-null     int64         
 1   Date                 88 non-null     datetime64[us]
 2   Day                  88 non-null     int32         
 3   Month                88 non-null     str           
 4   Year                 88 non-null     int64         
 5   Customer_Age         88 non-null     int64         
 6   Age_Group            88 non-null     category      
 7   Customer_Gender      88 non-null     str           
 8   Country              88 non-null     str           
 9   State                88 non-null     str           
 10  Product_Category     88 non-null     str           
 11  Sub_Category         88 non-null     str           
 12  Product_Description  88 non-null     str           
 13  Order_Quantity       88 non-null     int64         
 

## 14. Export Cleaned Dataset

In [19]:
df.to_excel('Cleaned_bike_sales_data.xlsx', index=False)

## Summary

| Step | Before | After 
|---|---|---|
| Rows | *(fill in from Section 2)* | *(fill in from Section 13)* |
| Duplicate rows | *(fill in)* | 0 |
| Missing Unit_Cost / Unit_Price | *(fill in)* | 0 |
| Missing Product_Description | *(fill in)* | 0 |

**Key decisions:**
- Imputed missing cost/price with **product-level medians** rather than a global mean, to avoid distorting prices across dissimilar products.
- Standardized free-text categorical fields (gender, country, month) rather than dropping malformed rows, to preserve sample size.
- Bucketed ages into four marketing-standard groups to support downstream segmentation analysis.
